# Importing libraries

In [60]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [61]:
%matplotlib inline

In [62]:
import numpy as np
import pandas as pd
import math
import dill
from pathlib import Path
from sklearn.linear_model import LinearRegression

# Loading Datasets

### Kaggle S&P500 data
- | `sp500_stocks.csv.xz` | Daily open, high, low, close, adjusted close and volume for each stock |
- | `sp500_companies.csv` | One row per company: sector, industry, market cap, etc. |

In [63]:
df_stocks=pd.read_csv(f"data/sp500_stocks.csv.xz",parse_dates=['date'])
df_comp=pd.read_csv(f"data/sp500_companies.csv",parse_dates=['date_added'])

### FRED data
- | `VIXCLS` | VIX: the volatility the market expects over the next 30 days |
- | `DGS10` | 10-year US Treasury interest rate |
- | `T10Y2Y` | Difference between 10-year and 2-year rates (a classic recession signal) |

In [64]:
df_VIX=pd.read_csv(f"data/VIXCLS.csv",parse_dates=['observation_date'])
df_DGS=pd.read_csv(f"data/DGS10.csv",parse_dates=['observation_date'])
df_T10Y2Y=pd.read_csv(f"data/T10Y2Y.csv",parse_dates=['observation_date'])

# Presenting my data 

In [65]:
dfs = {'Stocks': df_stocks, 'Companies': df_comp, 'VIX': df_VIX,
       '10Y rate': df_DGS, '10Y-2Y spread': df_T10Y2Y}

for name, df in dfs.items():
    print(f"\n===== {name}: {df.shape[0]:,} rows x {df.shape[1]} columns =====")
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isnull().sum().sum())
    display(df.head(3))


===== Stocks: 2,949,733 rows x 7 columns =====
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'symbol']
Missing values: 0


,date,open,high,low,close,volume,symbol
0,2000-01-03,46.873742,46.985347,40.103091,42.855995,4674353.0,A
1,2000-01-04,40.549510,40.995929,38.540636,39.582275,4765083.0,A
2,2000-01-05,39.433469,39.582277,35.899340,37.126984,5758642.0,A



===== Companies: 503 rows x 7 columns =====
Columns: ['symbol', 'company', 'sector', 'sub_industry', 'headquarters', 'date_added', 'founded']
Missing values: 0


,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888



===== VIX: 6,979 rows x 2 columns =====
Columns: ['observation_date', 'VIXCLS']
Missing values: 218


,observation_date,VIXCLS
0,2000-01-03,24.21
1,2000-01-04,27.01
2,2000-01-05,26.41



===== 10Y rate: 6,979 rows x 2 columns =====
Columns: ['observation_date', 'DGS10']
Missing values: 288


,observation_date,DGS10
0,2000-01-03,6.58
1,2000-01-04,6.49
2,2000-01-05,6.62



===== 10Y-2Y spread: 6,980 rows x 2 columns =====
Columns: ['observation_date', 'T10Y2Y']
Missing values: 288


,observation_date,T10Y2Y
0,2000-01-03,0.20
1,2000-01-04,0.19
2,2000-01-05,0.24


# Combining Datasets

In [66]:
comp = df_comp[['symbol', 'sector', 'company', 'sub_industry']]
vix  = df_VIX[['observation_date', 'VIXCLS']]
dgs  = df_DGS[['observation_date', 'DGS10']]
t10  = df_T10Y2Y[['observation_date', 'T10Y2Y']]

df_raw = df_stocks.merge(comp, on='symbol', how='left')

for fred in [vix, dgs, t10]:
    df_raw = df_raw.merge(fred, left_on='date', right_on='observation_date', how='left')
    df_raw = df_raw.drop(columns='observation_date')

In [67]:
print(f"\n===== Full Dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns =====")
print("Columns:", df_raw.columns.tolist())
print("Missing values:", df_raw.isnull().sum().sum())
display(df_raw.head(3))


===== Full Dataset: 2,949,733 rows x 13 columns =====
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'symbol', 'sector', 'company', 'sub_industry', 'VIXCLS', 'DGS10', 'T10Y2Y']
Missing values: 43164


,date,open,high,low,close,volume,symbol,sector,company,sub_industry,VIXCLS,DGS10,T10Y2Y
0,2000-01-03,46.873742,46.985347,40.103091,42.855995,4674353.0,A,Health Care,Agilent Technologies,Life Sciences Tools & Services,24.21,6.58,0.20
1,2000-01-04,40.549510,40.995929,38.540636,39.582275,4765083.0,A,Health Care,Agilent Technologies,Life Sciences Tools & Services,27.01,6.49,0.19
2,2000-01-05,39.433469,39.582277,35.899340,37.126984,5758642.0,A,Health Care,Agilent Technologies,Life Sciences Tools & Services,26.41,6.62,0.24


In [68]:
%time df_raw.to_csv(f"data/full_dataset.csv", index = False)

CPU times: user 25.8 s, sys: 1.35 s, total: 27.1 s
Wall time: 28.8 s


# Data Preprocessing